#Call Library

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

ModuleNotFoundError: No module named 'plotly'

#Load Data

In [22]:
fact_sales = pd.read_csv('/content/fact_sales.csv')
dim_date = pd.read_csv('/content/dim_date.csv')
dim_product = pd.read_csv('/content/dim_product.csv')
dim_market = pd.read_csv('/content/dim_marketplace.csv')

df_merged = pd.merge(fact_sales, dim_product, on='Product_Key', how='left')
df_merged = pd.merge(df_merged, dim_date, on='Date_Key', how='left')
df_merged = pd.merge(df_merged, dim_market, on='Marketplace_Key', how='left')

# Convert 'Date' column to datetime immediately after merging
df_merged['Date'] = pd.to_datetime(df_merged['Date'], errors='coerce')

print(df_merged.head())

   Sales_Key  Date_Key  Marketplace_Key  Product_Key Account Title_x  Taxes  \
0          1         1                1            1     [ UK ] Aava -50.34   
1          2         2                1            1     [ UK ] Aava -58.89   
2          3         3                1            1     [ UK ] Aava -72.41   
3          4         4                1            1     [ UK ] Aava -41.43   
4          5         5                1            1     [ UK ] Aava -47.04   

   Orders  Units  Refunded  Refund %  ...  Day  Month  Month_Name  Quarter  \
0      17     17         0       0.0  ...    1     10     October        4   
1      20     20         0       0.0  ...    2     10     October        4   
2      18     24         0       0.0  ...    3     10     October        4   
3      13     14         0       0.0  ...    4     10     October        4   
4      16     16         0       0.0  ...    5     10     October        4   

   Year  Week_Of_Year  Day_Name  Is_Weekend  Market Plac

#Dashboard 1 - Overview

KPIs:
1. Total Revenue
2. Total Orders
3. Total Units Sold
4. Total Net Profit
5. Blended Net Margin
6. Average Net ROI
7. Total ppc cost
8. Total Tacos
9. Total Net Roi

Charts:
1. True Profitability Trend (Combo Chart)
2. Order & Unit Trajectory (Dual Line Chart)
3. Average Order Value per Month
4. TACoS vs. Net Margin (Dual Axis Line Chart)
5. Net Margin % per Marketplace
6. Cost Structure (Waterfall Analysis)
7. MoM Growth Velocity (Bar/Line Chart)

##KPIs

###Total Revenue

In [5]:
total_revenue = fact_sales['Revenue'].sum()
print("Total Revenue:", total_revenue)

Total Revenue: 3628133.7699999996


###Total Orders

In [6]:
total_orders = fact_sales['Orders'].sum()
print("Total Orders:", total_orders)

Total Orders: 151164


###Total Units Sold

In [7]:
total_units = fact_sales['Units'].sum()
print("Total Units Sold:", total_units)

Total Units Sold: 177177


###Total Net Profit

In [8]:
total_net_profit = fact_sales['Net Profit'].sum()
print("Total Net Profit:", total_net_profit)

Total Net Profit: 815690.23


###Total ppc cost

In [13]:
total_ppc_cost = fact_sales['PPC Cost'].sum()
print("Total PPC Cost:", total_ppc_cost)

Total PPC Cost: -203132.78


###Total Net Roi

In [52]:
total_net_roi = fact_sales['Net ROI'].sum()
print("Total Net ROI:", total_net_roi)

Total Net ROI: 3920.8005999999996


###Average Net ROI

In [54]:
avg_net_roi = fact_sales['Net ROI'].mean()
print("Average Net ROI:", avg_net_roi)

Average Net ROI: 0.294288118291676


###blended_net_margin

In [55]:
blended_net_margin = total_net_profit / total_revenue
print("Blended Net Margin:", blended_net_margin)

Blended Net Margin: 0.22482363708436254


###total_tacos

In [56]:
total_tacos = abs(total_ppc_cost) / total_revenue
print("Total Tacos:", total_tacos)

Total Tacos: 0.055988227798998716


##Charts

###True Profitability Trend (Combo Chart)

In [39]:
import plotly.graph_objects as go

# 'Date' column is already in datetime format from AOaOuShhNGe7

# Create 'Month_Year' column
df_merged['Month_Year'] = df_merged['Year'].astype(str) + '-' + df_merged['Month_Name']

# Sort the DataFrame by 'Date' to ensure correct chronological order for plotting
df_merged = df_merged.sort_values(by='Date')

# Prepare monthly data
monthly_data = df_merged.groupby('Month_Year', sort=False)[['Revenue', 'Net Profit']].sum().reset_index()

# Create bar trace for Revenue
bar_trace = go.Bar(
    x=monthly_data['Month_Year'],
    y=monthly_data['Revenue'],
    name='Total Revenue',
    marker_color='skyblue'
)

# Create line trace for Net Profit
line_trace = go.Scatter(
    x=monthly_data['Month_Year'],
    y=monthly_data['Net Profit'],
    mode='lines+markers',
    name='Net Profit',
    yaxis='y2',
    marker_color='red'
)

# Create figure with secondary y-axis
fig = go.Figure(data=[bar_trace, line_trace])

# Add titles and labels
fig.update_layout(
    title_text='True Profitability Trend: Revenue vs Net Profit',
    xaxis_title='Month and Year',
    yaxis_title='Total Revenue',
    yaxis2=dict(
        title='Net Profit',
        overlaying='y',
        side='right'
    ),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255, 255, 255, 0)', bordercolor='rgba(255, 255, 255, 0)')
)

fig.show()

###Order & Unit Trajectory (Dual Line Chart)

In [40]:
import plotly.graph_objects as go

daily_trajectory = df_merged.groupby('Date')[['Orders', 'Units']].sum().reset_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_trajectory['Date'],
    y=daily_trajectory['Orders'],
    mode='lines+markers',
    name='Orders'
))

fig.add_trace(go.Scatter(
    x=daily_trajectory['Date'],
    y=daily_trajectory['Units'],
    mode='lines',
    name='Units Sold',
    line=dict(dash='dash')
))

fig.update_layout(
    title='Order & Unit Trajectory',
    xaxis_title='Date',
    yaxis_title='Count',
    hovermode='x unified'
)

fig.show()

### Average Order Value per Month

In [41]:
# Calculate monthly revenue and orders
monthly_aov_data = df_merged.groupby('Month_Year', sort=False).agg(
    total_revenue=('Revenue', 'sum'),
    total_orders=('Orders', 'sum')
).reset_index()

# Calculate Average Order Value (AOV)
monthly_aov_data['Average Order Value'] = monthly_aov_data['total_revenue'] / monthly_aov_data['total_orders']

display(monthly_aov_data[['Month_Year', 'Average Order Value']])

# Plotting the Average Order Value trend using Plotly
fig = px.line(
    monthly_aov_data,
    x='Month_Year',
    y='Average Order Value',
    title='Average Order Value Trend Per Month',
    markers=True,
    color_discrete_sequence=['purple']
)

fig.update_layout(
    xaxis_title='Month and Year',
    yaxis_title='Average Order Value',
    xaxis_tickangle=-45,
    hovermode='x unified'
)

fig.show()

,Month_Year,Average Order Value
0,2020-October,24.657366
1,2020-November,24.550804
2,2020-December,24.854247
3,2021-January,23.398763
4,2021-February,23.037921
5,2021-March,23.375527
6,2021-April,24.419615


###TACoS vs. Net Margin (Dual Axis Line Chart)

In [42]:
import plotly.graph_objects as go

# Calculate monthly percentages
# Handle division by zero for Revenue to avoid 'inf' values
df_merged['TACoS'] = np.where(df_merged['Revenue'] != 0, abs(df_merged['PPC Cost']) / df_merged['Revenue'], 0)
df_merged['Net_Margin_Pct'] = np.where(df_merged['Revenue'] != 0, df_merged['Net Profit'] / df_merged['Revenue'], 0)

monthly_tacos = df_merged.groupby('Month_Year', sort=False)[['TACoS', 'Net_Margin_Pct']].mean().reset_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=monthly_tacos['Month_Year'],
    y=monthly_tacos['TACoS'],
    mode='lines+markers',
    name='TACoS %',
    marker_color='blue'
))

fig.add_trace(go.Scatter(
    x=monthly_tacos['Month_Year'],
    y=monthly_tacos['Net_Margin_Pct'],
    mode='lines+markers',
    name='Net Margin %',
    marker_color='green'
))

fig.update_layout(
    title='TACoS vs Net Margin Trend',
    xaxis_title='Month and Year',
    yaxis_title='Percentage',
    xaxis_tickangle=-45,
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255, 255, 255, 0)', bordercolor='rgba(255, 255, 255, 0)'),
    hovermode='x unified'
)

fig.show()

### Net Margin % per Marketplace

In [30]:
# Calculate Net Margin Percentage for each row, handling division by zero
df_merged['Net_Margin_Pct_Marketplace'] = np.where(df_merged['Revenue'] != 0, df_merged['Net Profit'] / df_merged['Revenue'], 0)

# Group by 'Market Place' and calculate the mean of 'Net_Margin_Pct_Marketplace'
net_margin_by_marketplace = df_merged.groupby('Market Place')['Net_Margin_Pct_Marketplace'].mean().reset_index()

display(net_margin_by_marketplace)

,Market Place,Net_Margin_Pct_Marketplace
0,DE,0.243986
1,ES,0.204599
2,FR,0.122173
3,IT,0.111746
4,UK,0.131384


In [46]:
import plotly.express as px

# Calculate Net Margin Percentage for each row, handling division by zero
df_merged['Net_Margin_Pct_Marketplace'] = np.where(df_merged['Revenue'] != 0, df_merged['Net Profit'] / df_merged['Revenue'], 0)

# Group by 'Market Place' and calculate the mean of 'Net_Margin_Pct_Marketplace'
net_margin_by_marketplace = df_merged.groupby('Market Place')['Net_Margin_Pct_Marketplace'].mean().reset_index()

# Plotting with Plotly Express
fig = px.bar(
    net_margin_by_marketplace,
    x='Market Place',
    y='Net_Margin_Pct_Marketplace',
    title='Net Margin Percentage Across Marketplaces',
    color='Market Place',
    color_discrete_sequence=px.colors.qualitative.Plotly
)

fig.update_layout(
    xaxis_title='Market Place',
    yaxis_title='Net Margin Percentage',
    yaxis_range=[0, net_margin_by_marketplace['Net_Margin_Pct_Marketplace'].max() * 1.1] # Set y-axis limit slightly above max for better visualization
)

fig.show()

###Cost Structure (Waterfall Analysis)

In [47]:
import plotly.express as px

# Aggregate costs
costs = ['COGS', 'FBA Fees', 'Taxes', 'PPC Cost']
values = [abs(df_merged[c].sum()) for c in costs]

# Create a DataFrame for Plotly
cost_df = pd.DataFrame({'Cost Type': costs, 'Amount': values})

# Plotting with Plotly Express
fig = px.bar(
    cost_df,
    x='Cost Type',
    y='Amount',
    title='Cost Structure Analysis',
    color='Cost Type',
    color_discrete_sequence=px.colors.qualitative.Plotly
)

fig.update_layout(
    xaxis_title='Cost Type',
    yaxis_title='Amount'
)

fig.show()

###MoM Growth Velocity (Bar/Line Chart)

In [45]:
import plotly.express as px

monthly_revenue = df_merged.groupby('Month_Year')['Revenue'].sum().reset_index()
monthly_revenue['MoM_Growth'] = monthly_revenue['Revenue'].pct_change() * 100

fig = px.bar(
    monthly_revenue,
    x='Month_Year',
    y='MoM_Growth',
    title='Monthly Revenue Growth (%)',
    color_discrete_sequence=['teal']
)

fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="0% Growth", annotation_position="bottom right")

fig.update_layout(
    xaxis_title='Month and Year',
    yaxis_title='Growth (%)',
    xaxis_tickangle=-45,
    hovermode='x unified'
)

fig.show()